### Homework

In [ ]:
!pip install gdown
!gdown --id 19FYxTXMop51l6aBpdAOoq-LZiSOuK6Qn --output alt_syllogisms.csv

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=19FYxTXMop51l6aBpdAOoq-LZiSOuK6Qn
To: /content/alt_syllogisms.csv
100% 89.8k/89.8k [00:00<00:00, 3.64MB/s]


In [ ]:
!ls -lh alt_syllogisms.csv

-rw-r--r-- 1 root root 88K Feb  6  2024 alt_syllogisms.csv


In [ ]:
import pandas as pd
df_syll = pd.read_csv("alt_syllogisms.csv")

In [ ]:
df_syll.head()

,Unnamed: 0,Id,Type,Mode,Text,Valid,Result
0,0,1,aaa,1,All flowers are plants. \nAll roses are flower...,Yes,1
1,1,2,aaa,1,All mammals are warm-blooded animals. \nAll do...,Yes,1
2,2,3,aaa,1,All people are mortals. \nAll Greeks are peopl...,Yes,1
3,3,4,aae,1,All flowers are plants. \nAll roses are flower...,No,0
4,4,5,aae,1,All mammals are warm-blooded animals. \nAll do...,No,0


This dataset has a combination of existing 256 types of syllogisms. Only 24 of them are valid, it means that according to the logical rules the conclusion is valid.

Compare the performance of the Logic dataset. You will use the Text as input and the Valid field as the value to predict.

You will compute a benchmark using the following models: BART and RoBERTa. You can usethe same code for the two models, ensure to delete models from the .cache folder in case you are running out of memory.

Ensure to use the datasets library to load the data. You can use sklearn to split the dataset in train and test splits.

In [ ]:
# Clean + map "Valid" to integers
valid_map = {
    "yes": 1, "no": 0,
    "true": 1, "false": 0,
    "1": 1, "0": 0
}
df["Valid"] = (
    df["Valid"]
    .astype(str).str.strip().str.lower()
    .map(valid_map)
)

# Sanity check for any unmapped values
if df["Valid"].isna().any():
    bad_vals = (
        pd.Series(pd.read_csv("alt_syllogisms.csv")["Valid"]
                  .astype(str).str.strip().str.lower().unique())
        .difference(pd.Series(list(valid_map.keys())))
    )
    raise ValueError(f"Unmapped Valid values found. Inspect these: {list(bad_vals)}")

df["Valid"] = df["Valid"].astype(int)
print(df["Valid"].value_counts(dropna=False))


Valid
0    696
1     72
Name: count, dtype: int64


In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

seed = 112358
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=seed, stratify=df["Valid"]
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

num_labels = 2
label2id = {"0": 0, "1": 1}
id2label = {0: "0", 1: "1"}

len(train_ds), len(test_ds), train_df["Valid"].mean(), test_df["Valid"].mean()


(614, 154, np.float64(0.09446254071661238), np.float64(0.09090909090909091))

Exercise 1. Compute the performance metrics (accuracy and f1-score) using the BART model (use the base model).

In [ ]:
# your code here

import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, AutoTokenizer, AutoModelForSequenceClassification

bart_name = "facebook/bart-base"
max_length = 256

tok_bart = AutoTokenizer.from_pretrained(bart_name, use_fast=True)

def preprocess_bart(batch):
    enc = tok_bart(batch["Text"], truncation=True, max_length=max_length, padding=False)
    enc["labels"] = [int(v) for v in batch["Valid"]]
    return enc

index_cols = [c for c in train_ds.column_names if c.startswith("__index_level_")]
keep_cols = ["Text", "Valid"] + index_cols

tok_train_bart = train_ds.map(preprocess_bart, batched=True,
                              remove_columns=[c for c in train_ds.column_names if c not in keep_cols]) \
                         .remove_columns(keep_cols)
tok_test_bart  = test_ds.map(preprocess_bart, batched=True,
                              remove_columns=[c for c in test_ds.column_names if c not in keep_cols]) \
                        .remove_columns(keep_cols)

from datasets import Value
tok_train_bart = tok_train_bart.cast_column("labels", Value("int64"))
tok_test_bart  = tok_test_bart.cast_column("labels",  Value("int64"))

tok_train_bart.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tok_test_bart.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model_bart = AutoModelForSequenceClassification.from_pretrained(
    bart_name,
    num_labels=2,
    id2label={0:"0",1:"1"},
    label2id={"0":0,"1":1},
)
if hasattr(model_bart.config, "use_cache"):
    model_bart.config.use_cache = False
model_bart.config.problem_type = "single_label_classification"

collator = DataCollatorWithPadding(tokenizer=tok_bart)

args = TrainingArguments(
    output_dir="./results_bart_base",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=25,
    report_to=[],
    seed=112358,
)

trainer_bart = Trainer(
    model=model_bart,
    args=args,
    train_dataset=tok_train_bart,
    eval_dataset=tok_test_bart,
    tokenizer=tok_bart,
    data_collator=collator,
)

trainer_bart.train()
# ---- Manual evaluation for BART (robust) ----
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bart.to(device)
model_bart.eval()

test_loader = DataLoader(
    tok_test_bart,
    batch_size=64,
    shuffle=False,
    collate_fn=collator,
)

all_logits = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model_bart(**batch)
        logits = out.logits.detach().cpu()
        all_logits.append(logits)
        all_labels.append(labels.cpu())

logits = torch.cat(all_logits, dim=0).numpy()   # (N, 2)
labels = torch.cat(all_labels, dim=0).numpy()   # (N,)

y_pred = logits.argmax(axis=1)
acc = accuracy_score(labels, y_pred)
f1m = f1_score(labels, y_pred, average="macro")

bart_results = {"model": "facebook/bart-base", "accuracy": float(acc), "f1_macro": float(f1m)}
print("BART-base metrics:", bart_results)





Map:   0%|          | 0/614 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/614 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/154 [00:00<?, ? examples/s]

Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-base and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-4106442091.py:58: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_bart = Trainer(


Step,Training Loss
25,0.398000
50,0.313100
75,0.313100
100,0.357700


BART-base metrics: {'model': 'facebook/bart-base', 'accuracy': 0.9090909090909091, 'f1_macro': 0.47619047619047616}


Exercise 2. Compute the performance metrics using the RoBERTa model (use the base model)

In [ ]:
# your code here

# ==== Exercise 2: RoBERTa-base on alt_syllogisms (Text -> Valid) ====
import numpy as np
import datasets as ds
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from sklearn.metrics import accuracy_score, f1_score

roberta_name = "roberta-base"
max_length = 256  # drop to 128 if OOM

tok_roberta = AutoTokenizer.from_pretrained(roberta_name, use_fast=True)

def preprocess_roberta(batch):
    enc = tok_roberta(batch["Text"], truncation=True, max_length=max_length, padding=False)
    enc["labels"] = [int(v) for v in batch["Valid"]]
    return enc

index_cols = [c for c in train_ds.column_names if c.startswith("__index_level_")]
keep_cols = ["Text", "Valid"] + index_cols

tok_train_rob = train_ds.map(preprocess_roberta, batched=True,
                             remove_columns=[c for c in train_ds.column_names if c not in keep_cols]) \
                        .remove_columns(keep_cols)
tok_test_rob  = test_ds.map(preprocess_roberta, batched=True,
                             remove_columns=[c for c in test_ds.column_names if c not in keep_cols]) \
                       .remove_columns(keep_cols)

tok_train_rob = tok_train_rob.cast_column("labels", ds.Value("int64"))
tok_test_rob  = tok_test_rob.cast_column("labels",  ds.Value("int64"))

tok_train_rob.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tok_test_rob.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model_rob = AutoModelForSequenceClassification.from_pretrained(
    roberta_name,
    num_labels=2,
    id2label={0:"0",1:"1"},
    label2id={"0":0,"1":1},
)

collator_rob = DataCollatorWithPadding(tokenizer=tok_roberta)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

args_rob = TrainingArguments(
    output_dir="./results_roberta_base",
    per_device_train_batch_size=16,      # lower to 8/4 if memory is tight
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",               # your env expects eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=25,
    report_to=[],
    seed=112358,
)

trainer_rob = Trainer(
    model=model_rob,
    args=args_rob,
    train_dataset=tok_train_rob,
    eval_dataset=tok_test_rob,
    tokenizer=tok_roberta,
    data_collator=collator_rob,
    compute_metrics=compute_metrics,
)

trainer_rob.train()
rob_metrics = trainer_rob.evaluate()
print("RoBERTa-base metrics:", rob_metrics)

roberta_results = {
    "model": roberta_name,
    "accuracy": float(rob_metrics.get("eval_accuracy", np.nan)),
    "f1_macro": float(rob_metrics.get("eval_f1_macro", np.nan)),
}
roberta_results


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-664394930.py:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_rob = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.360700,0.299420,0.909091,0.476190
2,0.313300,0.297132,0.909091,0.476190
3,0.328500,0.273533,0.909091,0.476190


RoBERTa-base metrics: {'eval_loss': 0.29941990971565247, 'eval_accuracy': 0.9090909090909091, 'eval_f1_macro': 0.47619047619047616, 'eval_runtime': 0.3458, 'eval_samples_per_second': 445.376, 'eval_steps_per_second': 8.676, 'epoch': 3.0}


{'model': 'roberta-base',
 'accuracy': 0.9090909090909091,
 'f1_macro': 0.47619047619047616}

Which model had the best performace?

# Your response here


*   Both BART-base and RoBERTa-base achieved ~0.91 accuracy and ~0.48 macro-F1, so no clear winner. The gap between accuracy and macro-F1 suggests class imbalance.
